# Sprint 1 — Run your completed local checkpoint

Author your service in VS Code. This notebook runs your exported local project in a separate environment; it does not teach or implement the assessment endpoint. Follow the Campus checkpoint brief and retain your local source as the working copy.

Before running, export `artifacts/sprint-1-checkpoint.zip` with `python -m tools.checkpoint export artifacts/sprint-1-checkpoint.zip`. Record `git rev-parse HEAD` and compute the ZIP's SHA-256 as shown in the lesson. Review the export for accidentally pasted secrets.

Save a copy of this notebook in Drive. Upload the ZIP using Colab's Files panel, then enter its path, your recorded source revision, and SHA-256 below. Setup clones current `main`, installs the course lock, and restores the archive onto that **fresh clone only**. It never overwrites your laptop project. The exported dependency metadata must match current main; a mismatch needs investigation rather than silently changing dependencies.

Deterministic checks need no provider key. The optional generated-answer check prepares a new index and uses provider quota. A working process and valid response shape do not establish answer quality.
The ZIP excludes your local `.env`. Set `CHECKPOINT_MODEL` below to the same approved `OPENROUTER_MODEL` you used locally (or keep the course default if you used it). A model name is configuration, not a secret; the provider key is entered only through the hidden prompt.


In [ ]:
from pathlib import Path
import hashlib, importlib, json, os, subprocess, sys, tempfile

ARCHIVE = Path("/content/sprint-1-checkpoint.zip")  # Your uploaded ZIP.
SOURCE_REVISION = ""  # Copy the full git rev-parse HEAD value recorded locally.
ARCHIVE_SHA256 = ""  # Copy the locally calculated digest; do not derive it here.
if not ARCHIVE.is_file():
    raise FileNotFoundError("Upload your local source export and set ARCHIVE.")
if len(SOURCE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in SOURCE_REVISION.lower()):
    raise ValueError("Enter the full source revision recorded at export time.")
if hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() != ARCHIVE_SHA256.strip().lower():
    raise ValueError("The uploaded archive does not match your recorded SHA-256.")
BASE = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
REPO = Path(tempfile.mkdtemp(prefix="fieldcare-checkpoint-", dir=BASE))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/richhiey/ai-app-dev_Mod-B.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO),
                "-c", str(REPO / "requirements.lock")], check=True)
# Editable-install hooks are not loaded retroactively by a running kernel.
# Select this fresh clone explicitly, including after a setup rerun.
previous_helper_path = globals().get("CHECKPOINT_HELPER_PATH")
if previous_helper_path in sys.path:
    sys.path.remove(previous_helper_path)
CHECKPOINT_HELPER_PATH = str(REPO / "src")
sys.path.insert(0, CHECKPOINT_HELPER_PATH)
for name in list(sys.modules):
    if name == "module_b" or name.startswith("module_b."):
        del sys.modules[name]
importlib.invalidate_caches()
from module_b.local_checkpoint import restore_local_checkpoint
restored = restore_local_checkpoint(ARCHIVE, REPO, ARCHIVE_SHA256)
RUNNER_REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Local export source revision:", SOURCE_REVISION)
print("Runner main revision:", RUNNER_REVISION)
print("Verified archive SHA-256:", restored["archive_sha256"])
print("Restored source files:", len(restored["files"]))
print("Ready. The runtime copy is separate from your local working project.")
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
CHECKPOINT_MODEL = DEFAULT_CHAT_MODEL  # Use the approved OPENROUTER_MODEL from your local .env.
if CHECKPOINT_MODEL not in CHAT_MODELS:
    raise ValueError("Choose an approved course model from CHAT_MODELS.")
print("Approved course models:", ", ".join(sorted(CHAT_MODELS)))
print("Checkpoint model:", CHECKPOINT_MODEL)


## Check your contract and preserve the original caller

Write your predictions in a text cell first. The short request has no equipment, so an accepted input should produce clarification. The `1000`-character request must still be accepted by v2; this catches an unchanged `600`-character practice limit. The `1001`-character request distinguishes the checkpoint's `1000`-character v2 limit from v1's `2000`-character agreement. An empty question should be rejected by either version.

The helper starts your restored `fieldcare.main:app` as a local HTTP server in this Colab runtime and waits for health before requests. Its context manager stops it afterwards. No connection to your laptop is used. A missing v2 path is incomplete submitted work; the notebook does not create it for you.

In [ ]:
import httpx
from module_b.runtime import ServiceProcess

observations = []
generation_attempted = False
short_body = {"question": "Which filter checks are documented?"}
boundary_body = {"question": "filter " + "x" * 993}
long_body = {"question": "filter " + "x" * 994}
with ServiceProcess(app="fieldcare.main:app", project_dir=REPO / "service", startup_timeout=60,
                    env={"OPENROUTER_API_KEY": "", "OPENROUTER_MODEL": CHECKPOINT_MODEL, "FIELDCARE_WORK_DIR": str(REPO / "var")}) as service:
    with httpx.Client(base_url=service.base_url, timeout=30) as client:
        print("Health:", client.get("/health").json())
        schema = client.get("/openapi.json").json()
        print("Registered paths:", sorted(schema["paths"]))
        for path in ("/v1/diagnose", "/v2/diagnose"):
            operation = schema["paths"].get(path, {}).get("post", {})
            print(path, "request declaration:", operation.get("requestBody", "MISSING"))
            for label, body in (("short, no equipment", short_body),
                                ("1000 characters, no equipment", boundary_body),
                                ("1001 characters, no equipment", long_body),
                                ("empty question", {"question": ""})):
                result = client.post(path, json=body)
                row = {"path": path, "case": label, "http_status": result.status_code, "body": result.json()}
                observations.append(row)
                print(json.dumps(row, indent=2))

## Optional: observe the supported answer

Enable this only when you are ready to use provider quota. The key comes from a private prompt, is passed to the preparation/server processes, and is not written into the source export. This cell prepares the current documents, then sends the checkpoint's supported request. It cannot establish that every claim is correct: compare one actual claim with its cited document afterwards.

Leave the cell disabled when provider access is unavailable and record that the generated-answer evidence is outstanding. The two calls check both the new operation and the original caller. Generated wording varies.
Both versions use `CHECKPOINT_MODEL`, selected in setup. Confirm it matches your local model setting before enabling this cell. Successful generated-answer evidence is part of the brief; leaving the switch off keeps that part outstanding.


In [ ]:
RUN_GENERATION = False
if RUN_GENERATION:
    generation_attempted = True
    from getpass import getpass
    provider_key = getpass("OpenRouter API key (hidden): ").strip()
    generation_env = {**os.environ, "OPENROUTER_API_KEY": provider_key,
                      "OPENROUTER_MODEL": CHECKPOINT_MODEL,
                      "FIELDCARE_WORK_DIR": str(REPO / "var"),
                      "PYTHONPATH": str(REPO / "service") + os.pathsep + str(REPO / "src")}
    try:
        subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"], cwd=REPO,
                       env=generation_env, check=True)
        body = {"question": "After fitting a replacement filter on HX-220, which seating and airflow checks should the supervisor confirm?",
                "equipment_id": "EQ-FC-1002"}
        with ServiceProcess(app="fieldcare.main:app", project_dir=REPO / "service", startup_timeout=60,
                            env={"OPENROUTER_API_KEY": provider_key, "OPENROUTER_MODEL": CHECKPOINT_MODEL, "FIELDCARE_WORK_DIR": str(REPO / "var")}) as service:
            with httpx.Client(base_url=service.base_url, timeout=120) as client:
                for path in ("/v1/diagnose", "/v2/diagnose"):
                    result = client.post(path, json=body)
                    row = {"path": path, "case": "supported checkpoint request", "http_status": result.status_code,
                           "body": result.json()}
                    observations.append(row)
                    print(json.dumps(row, indent=2))
    finally:
        provider_key = ""
        generation_env.pop("OPENROUTER_API_KEY", None)
else:
    print("Generation not attempted. Contract checks are separate evidence.")

## Interpret and save your evidence

Explain the actual results against your predictions. A `200` clarification proves an accepted contract/context path, not generation. A `422` identifies a rejected input. A `404` means the expected operation is absent. A provider failure must remain a failure in your record.

Use the Campus brief to judge completion; these observations are not an automatic grade. Save the notebook in Drive for your reasoning. The cell below writes selected synthetic-request observations and source provenance to a JSON file; download it from Colab's Files panel after reviewing it. Keep the original local source ZIP, since it identifies exactly the work you submitted. Continue editing in VS Code, not this runtime copy.

In [ ]:
EVIDENCE = BASE / ("sprint-1-evidence-" + REPO.name + ".json")
record = {"local_source_revision": SOURCE_REVISION, "runner_main_revision": RUNNER_REVISION,
          "archive_sha256": restored["archive_sha256"], "configured_model": CHECKPOINT_MODEL,
          "generation_attempted": generation_attempted, "observations": observations}
with EVIDENCE.open("x", encoding="utf-8") as output:
    json.dump(record, output, indent=2)
print("Review and download:", EVIDENCE)
print("Keep your original local source ZIP and save this notebook separately.")

## Recovery

- Upload/path/digest mismatch: verify the ZIP and recorded SHA-256 on your computer, then upload that exact file.
- Dependency mismatch: compare the exported source revision with runner main. Keep your source; resolve the course dependency difference before rerunning.
- Setup rerun: a fresh temporary clone is created every time. Earlier runtime copies and your original ZIP remain untouched.
- `404` or wrong contract: repair your route in VS Code, rerun local checks, and export a new ZIP with a new digest.
- Provider/index failure: preserve the actual failure and check configuration, quota, connectivity, or storage. Do not substitute a sample answer.
- Evidence file already exists: retain it; use a new filename for a new observation set.

## Summary

This checkpoint runs your exported service and records its contract and generation observations. The implementation and reasoning remain yours; the notebook supplies a separate execution environment, not a finished solution.